# Week 4, Notebook 4: Predicting Oil Prices

**Forecasting the price of Trinidad and Tobago crude oil.**

*Caribbean AI - Adrian Dunkley*

---

### What you will build
- A neural network that predicts the **monthly oil price** for Trinidad and
  Tobago, the Caribbean's biggest energy producer.
- A comparison against **linear regression**, to see when a network is worth it.
- A simple experiment to find **which clue matters most**.

### The big idea
Oil is the lifeblood of Trinidad and Tobago's economy. Unlike a stock price
(nearly random, last notebook), oil prices respond to **real, measurable
forces**: global demand, how much the country pumps, the strength of the US
dollar, and hurricane-season supply worries. When the data carries a real
signal, a neural network can genuinely predict it. Let us prove it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

oil = pd.read_csv("data/tt_oil_prices.csv", parse_dates=["month"])
print("Months of data:", len(oil))
oil.head()

### The clues we get each month

| Column | What it means |
| --- | --- |
| `prev_month_price` | Last month's oil price (prices move slowly, so this is a strong hint). |
| `global_demand_index` | World appetite for oil (100 = normal). Higher pushes price up. |
| `tt_production_kbd` | Trinidad's output in thousands of barrels per day. |
| `usd_index` | Strength of the US dollar. A strong dollar tends to lower oil prices. |
| `hurricane_season` | 1 in Aug-Oct, when storms threaten Caribbean supply. |
| `oil_price_usd` | The price we want to predict (US$ per barrel). |

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(oil["month"], oil["oil_price_usd"])
plt.title("Trinidad and Tobago crude oil price, 20 years (monthly)")
plt.xlabel("month"); plt.ylabel("US$ per barrel"); plt.grid(alpha=0.3)
plt.show()

### Step 1: Set up inputs, split, and scale

The same reliable routine: choose features, split into train and test **by
time** (oil is a time series, so no shuffling), and scale the features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ["prev_month_price", "global_demand_index", "tt_production_kbd",
            "usd_index", "hurricane_season"]
X = oil[features].values
y = oil["oil_price_usd"].values

# Time series: train on the earlier months, test on the later ones.
cut = int(len(X) * 0.8)
X_train, X_test = X[:cut], X[cut:]
y_train, y_test = y[:cut], y[cut:]

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train months:", len(X_train), " Test months:", len(X_test))

### Step 2: A baseline and a linear model to beat

Two things to beat:

1. **Naive baseline**: predict this month's price = last month's price.
2. **Linear regression**: a straight-line model. If a network cannot beat a
   simple line, the extra complexity is not worth it.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# 1) Naive baseline
naive_pred = X_test[:, 0]                 # prev_month_price column
naive_mae = mean_absolute_error(y_test, naive_pred)

# 2) Linear regression
linreg = LinearRegression().fit(X_train_s, y_train)
lin_pred = linreg.predict(X_test_s)
lin_mae = mean_absolute_error(y_test, lin_pred)

print(f"Naive baseline MAE: US${naive_mae:.2f}")
print(f"Linear model MAE:   US${lin_mae:.2f}  (R2 = {r2_score(y_test, lin_pred):.2f})")

### Step 3: Build and train the neural network

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(1)

model = keras.Sequential([
    layers.Input(shape=(len(features),)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(X_train_s, y_train, validation_split=0.15,
                    epochs=200, batch_size=16, verbose=0)

plt.figure(figsize=(7, 4))
plt.plot(history.history["mae"], label="training")
plt.plot(history.history["val_mae"], label="validation")
plt.title("Learning curve (MAE)"); plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
nn_pred = model.predict(X_test_s, verbose=0).flatten()
nn_mae = mean_absolute_error(y_test, nn_pred)
nn_r2 = r2_score(y_test, nn_pred)

print("--- Test-set results (unseen months) ---")
print(f"Naive baseline MAE: US${naive_mae:.2f}")
print(f"Linear model MAE:   US${lin_mae:.2f}")
print(f"Neural net MAE:     US${nn_mae:.2f}   (R2 = {nn_r2:.2f})")
print("\nThe network explains most of the price swings - real signal, unlike the stock!")

In [ ]:
# Plot the network's forecast against the real price on the test period.
test_months = oil["month"].values[cut:]
plt.figure(figsize=(11, 4))
plt.plot(test_months, y_test, label="real price", linewidth=2)
plt.plot(test_months, nn_pred, label="network forecast", alpha=0.85)
plt.title("T&T oil price: network forecast vs reality (unseen months)")
plt.xlabel("month"); plt.ylabel("US$ per barrel"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

### Step 4: Which clue matters most?

A neat trick to rank a feature's importance: **shuffle** that one column in the
test set and see how much the error grows. If scrambling a clue barely changes
the error, the model was not leaning on it. If the error jumps, that clue was
doing heavy lifting. This is called **permutation importance**.

In [ ]:
rng = np.random.default_rng(0)
base_error = mean_absolute_error(y_test, model.predict(X_test_s, verbose=0).flatten())

importances = []
for j, name in enumerate(features):
    X_shuffled = X_test_s.copy()
    rng.shuffle(X_shuffled[:, j])          # scramble just this clue
    err = mean_absolute_error(y_test, model.predict(X_shuffled, verbose=0).flatten())
    importances.append(err - base_error)   # how much worse without this clue

order = np.argsort(importances)
plt.figure(figsize=(8, 4))
plt.barh([features[i] for i in order], [importances[i] for i in order])
plt.title("Which clue matters most (bigger = more important)")
plt.xlabel("increase in error when the clue is scrambled")
plt.tight_layout()
plt.show()

### What you learned

- When data carries a **real signal** (oil responds to demand, production, the
  dollar, and hurricanes), a neural network predicts it well, unlike a random
  stock price.
- Always line the network up against a **naive baseline** and a **linear model**.
  Reach for the network when it clearly beats both.
- **Permutation importance** reveals which clues the model actually uses.

### Try it yourself
1. Remove `prev_month_price` from the features and retrain. How much harder does
   the job become?
2. Predict a made-up future month: high global demand, low production, hurricane
   season on. What price does the network expect?
3. Compare the feature-importance chart before and after removing a clue.